# Task 3 — MixUp 0.20, all five folds

Train **five fresh Gender models in Colab**, one for each saved validation fold **0–4**.
Each model trains on the other four folds for **30 epochs**, using **MixUp 0.20 without SAM**.
Keep the tested GeM CNN, image transforms and corrected labels.

Select a fresh **Colab L4 GPU**. The notebook and new source files must be on the GitHub branch below before **Run all**.
Reuse `MyDrive/MLA2/data/task3-data.zip`; no new data upload is needed.
Allow about **45 minutes for training**, plus data checks and final scoring. Actual time depends on the runtime.


## 1. Mount Drive and select the repository

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task3-mixup-five-fold-training"
REPO_DIR = Path("/content/MLA2")
DRIVE_PROJECT = Path("/content/drive/MyDrive/MLA2")
DATA_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
drive.mount("/content/drive", force_remount=False)

## 2. Fetch code from GitHub

Keep local edits safe: repository updates use a fast-forward merge.

In [ ]:
def run_checked(command):
    return subprocess.run([str(x) for x in command], check=True)


if (REPO_DIR / ".git").is_dir():
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError("The local checkout belongs to another repository")
    run_checked(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "switch", BRANCH])
    run_checked(["git", "-C", REPO_DIR, "merge", "--ff-only", f"origin/{BRANCH}"])
else:
    run_checked(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
print("Code commit:")
run_checked(["git", "rev-parse", "HEAD"])

## 3. Reuse the existing teacher images

Read the saved development rows and extract only missing images from the Drive ZIP.
Code and splits come from GitHub. The runner checks all image hashes before training.


In [ ]:
import pandas as pd

splits = pd.read_csv(REPO_DIR / "data/processed/splits.csv", keep_default_na=False)
paths = splits.loc[splits.partition.eq("development"), "path"].tolist()
missing = [source_path for source_path in paths if not (REPO_DIR / source_path).is_file()]
if missing:
    local_zip = Path("/content/task3-data.zip")
    shutil.copyfile(DATA_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as archive:
        for source_path in missing:
            if not source_path.startswith("data/raw/teacher/train/images_train/"):
                raise ValueError(f"Unexpected development image path: {source_path}")
            target = (REPO_DIR / source_path).resolve()
            if not target.is_relative_to(REPO_DIR.resolve()):
                raise ValueError("Image path leaves the repository")
            target.parent.mkdir(parents=True, exist_ok=True)
            partial = target.with_suffix(target.suffix + ".partial")
            with archive.open(source_path) as source, partial.open("wb") as output:
                shutil.copyfileobj(source, output)
            partial.replace(target)
print(f"Development images ready: {len(paths):,}")

## 4. Check the fixed recipe and Colab runtime

GeM p=3 CNN, dropout **0.30**, MixUp **0.20**, and ordinary AdamW updates.
Learning rate **0.001**, minimum **0.00001**, weight decay **0.0001**, batch **128**, seed **2753**.
Keep translation ±2 px (50%), mild darkening (25%) and grayscale (10%).
Each fold uses normalization fitted only on its own training images.
Save epoch **30**, with cosine `T_max=30`; validation never chooses an epoch.

Match the recorded Colab stack in `requirements/colab-task3-runtime.txt`.
The check stops before training if the runtime differs. Use a compatible runtime instead of installing the local project’s newer PyTorch requirements.


In [ ]:
import json

import torch

from fashion.train.task3_baseline import runtime_environment, validate_verified_colab_runtime
from fashion.train.task3_gender_mixup_cv import EXPERIMENT, SOURCE_CONFIG

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab L4 GPU runtime before training")
validate_verified_colab_runtime(runtime_environment(torch.device("cuda")))
saved = json.loads((REPO_DIR / SOURCE_CONFIG).read_text())
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
print("Folds: 0, 1, 2, 3, 4; epochs:", saved["epochs"], "; SAM: disabled")
print("Output:", DRIVE_TASK_DIR / "experiments" / EXPERIMENT / "gender")


## 5. Train and save all five folds

Every fold enters the shared Drive registry before training; a local registry copy is kept too.
Each epoch saves mixed training loss and validation scores. Final clean training, validation and five corruption checks use IEEE FP32, matching the SAM25 comparison.

Run all again to verify and reuse completed folds. An interrupted fold starts again from fresh weights in a new folder.
Keep only one active Colab session writing this experiment. Old screen runs and the saved final model stay separate.


In [ ]:
from fashion.train.task3_gender_mixup_cv import run_gender_mixup_cv

result = run_gender_mixup_cv(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
)
print("Status:", result["status"])
print("Completed folds:", result["folds"])
print("Validation images:", result["oof_rows"])
print("Pooled validation macro-F1:", f"{result['pooled_oof']['macro_f1']:.2%}")
print("Summary:", result["summary_path"])


## 6. Review the saved result

The pooled OOF file has one validation prediction per development image, from the model that did not train on it.
Do not average the five models to score development images: four models trained on each image.

Each fold folder contains its weights, normalization, config, 30-epoch history, MixUp receipt, row lists, clean predictions and corrupted-image predictions.
`cv_summary.json` links all five models and gives the pooled score. `fold_summary.csv` gives each fold’s score and clean training gap; `robustness.csv` gives the pooled corruption scores.
These are development results after earlier screens, not a new blind test. Compare with SAM25 before making a final-model decision.
This notebook does not evaluate holdout/test images or change the accepted final model.


In [ ]:
from IPython.display import display

output = Path(result["summary_path"]).parent
folds = pd.read_csv(output / "fold_summary.csv")
assert sorted(folds.validation_fold.tolist()) == [0, 1, 2, 3, 4]
assert result["oof_rows"] == 32773
for model in result["models"]:
    history = pd.read_csv(output / model["directory"] / "history.csv")
    assert history.epoch.tolist() == list(range(1, 31))
    assert history.selected_checkpoint.tolist() == [False] * 29 + [True]
display(folds)
display(pd.read_csv(output / "robustness.csv"))
print("Saved predictions:", output / "oof_predictions.csv")
